# TD5 Partie 1 : Raisonnement avec SWRL

Le raisonnement par règles (SWRL) permet d'**inférer** de nouveaux faits
à partir de faits existants. Contrairement à une requête SPARQL qui cherche
ce qui est déjà là, une règle SWRL **crée** de nouveaux triplets.

On va appliquer deux règles :
1. Sur `family.owl` : toute personne de plus de 60 ans est une `oldPerson`
2. Sur notre KB musicale : si A est influencé par B et B est influencé par C, alors A est influencé par C

### Installation

In [3]:
!conda install -c conda-forge owlready2 -y

3 channel Terms of Service accepted
Retrieving notices: - \ done
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: - \ | / - \ | done

## Package Plan ##

  environment location: C:\Users\sandy\anaconda3\envs\rstudio

  added / updated specs:
    - owlready2


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.2.25  |       h4c7d964_0         144 KB  conda-forge
    owlready2-0.39             |   py37h51bd9d9_0        20.3 MB  conda-forge
    python_abi-3.7             |          2_cp37m           4 KB  conda-forge
    ------------------------------------------------------------
                                           Total:        20.4 MB

The following NEW packages will be INSTALLED:

  owlready2          conda-forge/win-64::owlready2-0.39-py37h51bd9d9_0 
  python_abi         conda-forge/win-64::python_abi-3.7-2_cp37m 

Th



==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda




### Chargement de family.owl

Le fichier contient une ontologie familiale avec 12 individus
et leurs relations (isParentOf, isMarriedWith, isSiblingOf...).
Chaque individu a un attribut `age` qui nous servira pour la règle SWRL.

In [4]:
from owlready2 import *

CHEMIN = "C:/Users/sandy/OneDrive/Desktop/Cours A4/Web Datamining/TD4_Project/"

# Chargement de l'ontologie
onto = get_ontology(CHEMIN + "family_lab.owl").load()

print("Ontologie chargee.")
print(f"Namespace : {onto.base_iri}")
print()

# On identifie les namespaces
FAM = onto.get_namespace("http://www.owl-ontologies.com/family_lab.owl#")
ONT = onto.get_namespace("http://www.owl-ontologies.com/unnamed.owl#")

# Affichage de tous les individus et leur age
print("Individus dans l'ontologie :")
print(f"{'Nom':<12} {'Age':>5} {'Type'}")
print("-" * 35)
for individu in onto.individuals():
    nom  = individu.name
    age  = individu.age if hasattr(individu, 'age') and individu.age else "?"
    types = [c.name for c in individu.is_a if hasattr(c, 'name')]
    print(f"{nom:<12} {str(age):>5}  {', '.join(types)}")

Ontologie chargee.
Namespace : http://www.owl-ontologies.com/family_lab.owl#

Individus dans l'ontologie :
Nom            Age Type
-----------------------------------
Alex            25  Female
Thomas          40  Male
Michael          5  Male
Tom             10  Male
Chloe           18  Female
Marie           69  Female
Peter         70.0  Male
Claude           5  Female
Sylvie          30  Female
John          45.0  Male
Pedro           10  Male
Paul            38  Male


### Définition de la classe oldPerson

Avant d'écrire la règle, on doit créer la classe `oldPerson` dans l'ontologie.
C'est la classe que le raisonneur va peupler automatiquement.

In [6]:
with onto:
    # On cree la classe oldPerson comme sous-classe de Person
    class oldPerson(ONT.Person):
        pass

print("Classe oldPerson creee")
print(f"oldPerson est une sous-classe de : {oldPerson.is_a}")

# Combien d'oldPerson avant la regle ?
avant = list(onto.individuals())
print(f"\noldPerson avant la regle SWRL : {len(list(oldPerson.instances()))} individu(s)")

Classe oldPerson creee
oldPerson est une sous-classe de : [unnamed.Person]

oldPerson avant la regle SWRL : 0 individu(s)


### Règle SWRL 1 : oldPerson

La règle se lit comme une implication logique :
```
SI   p est une Person
ET   p a un age a
ET   a > 60
ALORS  p est une oldPerson
```

En notation SWRL formelle :
$$\text{Person}(p) \wedge \text{age}(p, a) \wedge \text{swrlb:greaterThan}(a, 60) \Rightarrow \text{oldPerson}(p)$$

In [11]:


print("Application de la regle SWRL (implementation manuelle) :")
print()

with onto:
    count = 0
    for individu in ONT.Person.instances():
        age_val = individu.age
        if age_val and int(age_val) > 60:
            individu.is_a.append(oldPerson)
            print(f"  INFERE : {individu.name} est une oldPerson (age={age_val})")
            count += 1

print()
print(f"Total oldPerson detectes : {count}")
print()
print("Verification :")
for p in oldPerson.instances():
    print(f"  {p.name} (age={p.age})")

Application de la regle SWRL (implementation manuelle) :

  INFERE : Peter est une oldPerson (age=70.0)
  INFERE : Marie est une oldPerson (age=69)

Total oldPerson detectes : 2

Verification :
  Marie (age=69)
  Peter (age=70.0)


### Ce qu'on observe

Le raisonneur a identifié 2 oldPerson :
- **Marie** (69 ans) : 69 > 60 → condition satisfaite
- **Peter** (70 ans) : 70 > 60 → condition satisfaite

Les autres individus (Thomas 40 ans, Sylvie 30 ans, John 45 ans...)
n'ont pas été classifiés car leur âge est inférieur ou égal à 60.

Ce résultat n'existait pas dans le fichier original — il a été **inféré**
automatiquement par le raisonneur à partir de la règle qu'on a définie.

### Règle SWRL 2 : sur notre KB musicale

On va maintenant appliquer la même logique sur notre KB musicale.
La règle : si A est influencé par B, et B est influencé par C,
alors A est influencé par C (transitivité de l'influence).

$$\text{influencedBy}(A, B) \wedge \text{influencedBy}(B, C) \Rightarrow \text{influencedBy}(A, C)$$



In [12]:
from owlready2 import *
from rdflib import Graph, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, OWL

CHEMIN = "C:/Users/sandy/OneDrive/Desktop/Cours A4/Web Datamining/TD4_Project/"

# On charge notre KB alignee (plus legere que la KB expandee)
g = Graph()
g.parse(CHEMIN + "aligned_kb.ttl", format="turtle")

EXO = Namespace("http://musickg.example.org/ontology/")
EX  = Namespace("http://musickg.example.org/resource/")

# Affichage des relations influencedBy existantes
print("Relations influencedBy dans notre KB :")
for s, p, o in g.triples((None, EXO.influencedBy, None)):
    s_nom = str(s).split("/")[-1].replace("_", " ")
    o_nom = str(o).split("/")[-1].replace("_", " ")
    print(f"  {s_nom:<25} -> influencedBy -> {o_nom}")

Relations influencedBy dans notre KB :
  Amy Winehouse             -> influencedBy -> Nina Simone
  Kendrick Lamar            -> influencedBy -> Tupac Shakur
  Gorillaz                  -> influencedBy -> The Beatles
  Nirvana                   -> influencedBy -> The Beatles
  Portishead                -> influencedBy -> Massive Attack
  Radiohead                 -> influencedBy -> Pink Floyd
  The Beatles               -> influencedBy -> Elvis Presley


In [13]:
# Application manuelle de la regle de transitivite
# (owlready2 ne charge pas facilement un fichier .ttl, donc on applique la regle en Python)

print("Application de la regle de transitivite :")
print("influencedBy(A,B) ∧ influencedBy(B,C) → influencedBy(A,C)")
print()

nouvelles_relations = []

# On collecte toutes les relations influencedBy
influences = [(str(s), str(o)) for s, p, o in g.triples((None, EXO.influencedBy, None))]

# On cherche les chaines A->B->C
for (A, B) in influences:
    for (B2, C) in influences:
        if B == B2 and A != C:  # B est le meme, et on evite les boucles
            # Verifie que A->C n'existe pas deja
            if (A, C) not in influences:
                nouvelles_relations.append((A, B, C))

print(f"Nouvelles relations inferees : {len(nouvelles_relations)}")
print()
for A, B, C in nouvelles_relations:
    A_nom = A.split("/")[-1].replace("_", " ")
    B_nom = B.split("/")[-1].replace("_", " ")
    C_nom = C.split("/")[-1].replace("_", " ")
    print(f"  {A_nom:<25} influencedBy {C_nom}")
    print(f"    (via {B_nom})")
    print()

    # On ajoute le triplet infere dans le graphe
    g.add((URIRef(A), EXO.influencedBy, URIRef(C)))

Application de la regle de transitivite :
influencedBy(A,B) ∧ influencedBy(B,C) → influencedBy(A,C)

Nouvelles relations inferees : 2

  Gorillaz                  influencedBy Elvis Presley
    (via The Beatles)

  Nirvana                   influencedBy Elvis Presley
    (via The Beatles)





La règle de transitivité a inféré de nouvelles relations d'influence
qui n'étaient pas explicitement encodées dans notre KB.

Par exemple, **Nirvana influencedBy Elvis Presley** n'était pas dans notre
KB originale, mais la règle l'a déduit automatiquement :
- On sait que Nirvana influencedBy The Beatles
- On sait que The Beatles influencedBy Elvis Presley
- Donc : Nirvana influencedBy Elvis Presley

C'est la puissance du raisonnement : on peut extraire de nouvelles
connaissances sans avoir à les entrer manuellement.